In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats
import plotly.express as px
import plotly.graph_objects as go


FEATURE_PATH = "feature_autointerp_angled_2.5flash.csv"
TITLE = "gpt4o detector"

df = pd.read_csv(FEATURE_PATH)

In [ ]:
def plot_hist(df: pd.DataFrame, col: str, title: str):
    plt.figure(figsize=(10, 6))
    sns.histplot(df[col], bins=30, kde=True)

    plt.title(f'Distribution of {col.capitalize()} Values ({title})')
    plt.xlabel(col.capitalize())
    plt.ylabel('Frequency (Count)')

    plt.xlim(-1.0, 1.0)
    plt.axvline(0, color='red', linestyle='--', linewidth=1.5)
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()

plot_hist(df, 'correlation', TITLE)

In [ ]:
HOVER_DATA = ['terse_description', 'feature_index', 'human_density', 'target_density']
HOVER_TEMPLATE = (
    "<b>%{customdata[1]}:</b> %{customdata[0]}<br>"
    "LOR: %{x:.3f} (%{customdata[2]} :: %{customdata[3]})<br>"
    "Correlation: %{y:.3f}<extra></extra>"
)

def interactive_scatter_plot(df, x_col: str, y_col: str, title: str, include_trend_line: bool = False):
  title_text = f"{title.capitalize()}: {x_col.capitalize()} vs. {y_col.capitalize()}"
  figure_args = {'x': x_col, 'y': y_col, 'hover_data': HOVER_DATA, 'title': title_text}

  annotation_text = ""
  if include_trend_line:
      mask = df[x_col].notna() & df[y_col].notna()
      slope, _, r_value, _, _ = stats.linregress(df[mask][x_col], df[mask][y_col])
      annotation_text = f"R² = {(r_value**2):.3f}<br>Slope = {slope:.3f}"
      figure_args |= {'trendline': 'ols'}

  fig = px.scatter(df, **figure_args)
  fig.update_traces(hovertemplate=HOVER_TEMPLATE)
  fig.update_layout(xaxis_title=x_col.capitalize(), yaxis_title=y_col.capitalize())

  if include_trend_line:
      fig.update_traces(
          line=dict(dash='dot', color='red', width=2),
          hovertemplate=None,
          selector=dict(mode='lines')
      )
      fig.add_annotation(
          xref="paper", yref="paper",
          x=0.05, y=0.95,
          text=annotation_text,
          align="left",
          bgcolor="rgba(255, 255, 255, 0.8)",
          bordercolor="black",
      )

  fig.show()

interactive_scatter_plot(df, 'correlation', 'density', TITLE, include_trend_line=True)